***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 3 章：射电干涉测量中的位置天文学](3_0_introduction.ipynb)
    * 上一节：[3.4 方向余弦、相位中心与局部成像坐标](3_4_direction_cosine_coordinates.ipynb)
    * 下一节：[3.P 综合问题集](3_problem_set.ipynb)

***


## 3.5 时间标准、参考系统、参考架与精密天体测量

前几节建立了赤道坐标、时角、地平坐标和方向余弦之间的基本变换。常规成像软件通常自动完成这些转换，但高频相位参考、长基线干涉、宽视场成像和精密天体测量要求进一步明确时间标准、天球与地球参考系统、参考架实现以及地球定向参数。任何不一致的时间或几何约定都可能转化为延迟、可见度相位和图像位置的系统误差。

若基线向量为 $\mathbf b$，天源单位方向为 $\mathbf s$，远场平面波的几何延迟可写为

$$
\tau_g = {\mathbf{b}\cdot\mathbf{s}\over c}.
$$

当延迟模型存在误差 $\Delta\tau$ 时，频率 $\nu$ 处的相位误差幅度为

$$
|\Delta\phi| = 2\pi\nu|\Delta\tau|.
$$

相位正负号取决于相关器的指数约定和基线顺序，本节只讨论误差幅度。相位误差与频率成正比，因此皮秒量级的延迟误差在厘米波和毫米波观测中即可产生可测相位，纳秒量级误差在 GHz 频段可能跨越多个相位周期。精密天体测量必须将几何、时钟和传播介质引起的残余延迟控制在目标角位置精度所允许的范围内。

![时间标准、参考系统、参考架与几何模型](figures/time_reference_chain.png)

**图 3.5.1**：时间标准、参考系统、参考架与干涉几何之间的关系。UTC 用于观测记录和调度，TAI 与 TT 提供连续原子时和地球时，UT1 与地球定向参数确定地球相对于天球参考系统的姿态。源目录参考架、台站参考架和观测时刻必须共同进入延迟模型。

### 精度层级与适用范围

坐标与时间变换可用于概念教学、常规成像和精密天体测量，但三个层级所需的输入数据、验证方法和可支持结论不同。相位容差随频率、基线长度和动态范围而变化，因而不能给出适用于所有观测的统一角度门限。下表列出各层级至少需要满足的条件。

| 层级 | 目标 | 可以采用 | 必须补入 | 可支持的结论 |
|:---|:---|:---|:---|:---|
| 概念级 | 理解赤经/赤纬、地方恒星时/时角、方位角/高度角和 $(l,m,n)$ 的几何关系 | 假设的参考历元、近似恒星时、球面三角和 WGS 84 米级示例 | 单位、方向、正负号和数量级核对 | 可说明可观测性、坐标变换链和相位标度；不能据此确定真实数据的绝对相位或天体位置 |
| 常规成像级 | 保持相关器元数据、$(u,v,w)$、相位中心和世界坐标系统（WCS）的一致性 | Astropy、ERFA、SOFA 等标准实现和阵列软件 | 观测历元、台站坐标、IERS EOP、频率定义、相位旋转记录和图像位置交叉核对 | 可在合成波束和校准误差允许范围内解释形态与相对位置；不足以直接支持亚毫角秒天体测量 |
| 精密天体测量级 | 解释差分延迟、VLBI 相位参考、视差和自行 | 经验证的专业延迟模型与条纹拟合（fringe fitting） | 版本化 ITRF/ICRF、台站速度和位移、EOP 的最终值或预测值标识、时钟、对流层/电离层、校准源结构及目标-校准源角距离误差预算 | 只有通过残余延迟与延迟率、独立校准源和多历元稳定性检验，才能报告绝对或差分天体位置 |

调用标准高精度软件是常规成像和精密天体测量的必要条件，但不能替代输入数据、参考架版本和误差预算的验证。概念级公式仍适合用于检查软件输出的符号、数量级和适用边界。

### 3.5.1 时间标准的物理作用

观测日志通常采用协调世界时（UTC），以便与民用时间、调度和归档系统对应。国际原子时（TAI）是连续的原子时标；地球时（TT）满足 $\mathrm{TT}=\mathrm{TAI}+32.184\,\mathrm{s}$，常用于地心星历和动力学计算；UT1 则反映地球实际自转。UTC 不是无闰秒的连续计数，插入闰秒时会出现 23:59:60，TAI-UTC 随闰秒发生阶跃变化，而 UT1-UTC 必须由国际地球自转与参考系服务（IERS）发布的地球定向参数提供。

地球自转角（Earth Rotation Angle，ERA）由 UT1 儒略日直接定义：

$$
\mathrm{ERA}=2\pi\left[0.7790572732640+1.00273781191135448\left(\mathrm{JD}_{\rm UT1}-2451545.0\right)\right]\pmod{2\pi}.
$$

格林尼治平恒星时（GMST）由 ERA 和岁差相关项确定，格林尼治视恒星时（GAST）进一步包含章动相关改正。二者与向东为正的台站经度相加，分别得到地方平恒星时和地方视恒星时。因此，时角关系

$$
H = \mathrm{LST}-\alpha
$$

中的 LST 不能由 UTC 钟表读数通过固定比例直接换算得到。

地球定向参数（Earth Orientation Parameters，EOP）至少包括 UT1-UTC、极移 $x_p,y_p$，以及天球中间极相对于模型的改正 $dX,dY$。标准软件利用 IERS 数据完成转换，但必须记录数据版本及其属于最终值还是预测值。若 EOP 缺失或长期使用预测值，同一基线会被投影到略有差异的 $(u,v,w)$ 坐标，并表现为残余延迟、延迟率或图像位置偏差。闰秒附近的时间解析必须交由经过验证的时间库处理，不能假设每个 UTC 日恒有 86400 秒。

对于连通阵列，时间和 EOP 误差主要影响高频长基线相位及精确相位中心；对于 VLBI，台站钟差和钟漂本身也是待估参数。条纹拟合中的残余延迟和延迟率可用于联合修正几何、时钟和传播介质模型的剩余误差，但拟合收敛并不能证明绝对天体测量参考架已经正确。

标准软件的调用不能替代元数据记录。可重复的处理流程至少应保存时间库和延迟模型版本、闰秒表与 EOP 来源、EOP 数据状态、参考架版本以及相位跟踪中心。

![延迟误差对相位和天体测量的影响](figures/delay_phase_astrometry_scale.png)

**图 3.5.2**：延迟误差对应的相位误差和角位置误差。频率越高，相位对延迟误差越敏感；基线越长，同一延迟误差对应的角位置误差越小。长基线可提供较高角分辨率，同时对几何和时钟模型提出更严格要求。

### 3.5.2 天球与地球参考系统、参考架及相位中心

星表中的赤经和赤纬只有在指定参考系统、参考架和历元后才具有完整意义。国际天球参考系统（ICRS）规定理想的天球坐标系统，国际天球参考架（ICRF）通过河外射电源的位置实现该系统；国际地球参考系统（ITRS）规定随地球固连的地球参考系统，ITRF2020 等国际地球参考架版本则通过台站坐标和速度实现该系统。地心天球参考系统（GCRS）与 ITRS 在观测时刻的转换需要岁差-章动、ERA、极移和 EOP。恒星方向还需考虑自行、视差、光行差和引力偏折，太阳系天体的时变方向则应由星历计算。

在教学和常规成像中，参考方向常写为 $(\alpha_0,\delta_0)$，并用 $(l,m,n)$ 表示邻近方向。相关器按照延迟跟踪中心消除名义几何延迟；若图像参考方向与延迟跟踪中心不同，必须执行一致的相位旋转，并同步更新 $(u,v,w)$ 坐标和世界坐标系统（WCS）。固定的微小位置偏差会产生随基线变化的相位坡度，并在图像中表现为位置偏移；若误差还随时间、频率或天线变化，则可能进一步造成图像展宽、位置漂移或方向相关残差。

精密位置测量要求目录方向、台站坐标、地球姿态、传播介质、仪器时钟和相关器模型采用相容的定义。即使不一致的模型仍能生成形态上合理的图像，所得绝对位置、相对位置或频率相关位置也可能失去明确的天体测量意义。

### 3.5.3 台站坐标：大地坐标、地心地固坐标与局部坐标

台站位置可用大地纬度 $\varphi$、经度 $\lambda$ 和椭球高 $h$ 表示，也可用地心地固直角坐标（Earth-centered, Earth-fixed，ECEF）表示。ECEF 的 $X$ 轴穿过赤道与零经度的交点，$Y$ 轴穿过赤道与东经 $90^\circ$ 的交点，$Z$ 轴指向约定地球北极。ECEF 是一类随地球转动的直角坐标表示，并不等同于某一具体参考架；使用坐标数值时仍需注明所采用的基准、参考架版本和历元。

大地纬度是参考椭球法线与赤道面的夹角，地心纬度是地心径向量与赤道面的夹角，除赤道和两极外二者不同。椭球高是沿参考椭球法线量取的高度，也不等同于相对于平均海平面的正高。

WGS 84 椭球可用于本书的米级几何示例，[ecef.py](ecef.py) 给出了大地坐标与 ECEF 坐标之间的双向转换。精密阵列和 VLBI 应采用注明版本与参考历元的 ITRF 台站坐标、速度和天线参考点，并按需要加入板块运动、固体地球潮汐、海潮负荷和大气负荷等位移改正。基线向量必须由同一参考架、同一历元的两台天线坐标作差得到。

在参考台站处，可将 ECEF 差向量 $\Delta\mathbf r=(\Delta X,\Delta Y,\Delta Z)^T$ 转换为局部东-北-天顶（east-north-up，ENU）坐标：

$$
\begin{pmatrix}E\\N\\U\end{pmatrix}=
\begin{pmatrix}
-\sin\lambda & \cos\lambda & 0\\
-\sin\varphi\cos\lambda & -\sin\varphi\sin\lambda & \cos\varphi\\
\cos\varphi\cos\lambda & \cos\varphi\sin\lambda & \sin\varphi
\end{pmatrix}
\begin{pmatrix}\Delta X\\\Delta Y\\\Delta Z\end{pmatrix}.
$$

其中，$\varphi$ 必须为大地纬度，经度采用向东为正。该变换矩阵为正交矩阵，因而保持向量长度。地心处的大地坐标没有定义；在地理两极，经度以及局部东、北方向依赖所选参考子午线，而天顶方向仍可定义。坐标转换程序应显式检验这些边界情形。


### 3.5.4 精密天体测量的误差预算

对于孤立、未分辨且信噪比较高的点源，热噪声导致的位置不确定度可用合成波束宽度和信噪比估算：

$$
\sigma_\theta \simeq {\theta_{\rm beam}\over 2\,\mathrm{SNR}}.
$$

该关系给出近似高斯波束条件下的统计误差量级，具体系数随波束形状、拟合方法和像素间相关噪声而变化，且应分别考虑合成波束长轴和短轴方向。相位参考观测还受到校准源位置、目标-校准源角距离、对流层和电离层残余路径、天线位置、频率相关源结构、核心位移、时间插值及方向相关误差的限制。提高信噪比可以降低热噪声误差，但不能按同样规律消除系统误差。

基线增长使 $\theta_{\rm beam}\sim\lambda/B$ 减小，从而提高角分辨率。沿基线投影方向的小相位或延迟误差对应的角位置误差近似为

$$
|\Delta\theta|\simeq{\lambda|\Delta\phi|\over2\pi B}={c|\Delta\tau|\over B}.
$$

相位参考的目标是使目标源与参考源之间的差分相位能够由稳定的几何和传播模型解释，而不是使每个可见度相位都接近零。电离层群延迟近似随 $\nu^{-2}$ 变化，相位效应近似随 $\nu^{-1}$ 变化；远离强吸收线时，中性大气路径可近似视为非色散，但同一路径误差产生的相位随频率增大。目标与参考源角距离过大时，两条视线不能充分共享传播误差；参考源若存在结构或核心位移，也会限制绝对位置精度。

#### 数值示例：$10^\circ$ 相位预算对应的误差尺度

对于完全沿投影基线方向的位置误差，有 $|\Delta\phi|=2\pi(B/\lambda)|\Delta\theta|$。若仅将 UT1 时间误差视为地球转角误差，则最不利投影满足 $|\Delta\tau|\lesssim B\omega_\oplus|\Delta t|/c$。下面反求三类观测在 $10^\circ$ 相位预算下允许的位置误差、几何延迟误差和 UT1 误差上限。其中 UT1 上限是最不利投影下的保守估计，不能作为调度时间戳或条纹拟合解的通用容差。


In [ ]:
import numpy as np

C_LIGHT = 299_792_458.0
OMEGA_EARTH = 7.2921150e-5
phase_limit = np.deg2rad(10.0)
cases = [
    ('cm connected', 36e3, 1.4e9),
    ('mm connected', 16e3, 100e9),
    ('VLBI', 5e6, 8.4e9),
]

print('case            position [mas]  delay [ps]  UT1 bound [s]')
for name, baseline, frequency in cases:
    angle = phase_limit * C_LIGHT / (2 * np.pi * frequency * baseline)
    angle_mas = np.rad2deg(angle) * 3.6e6
    delay_ps = phase_limit / (2 * np.pi * frequency) * 1e12
    ut1_bound = angle / OMEGA_EARTH
    recovered_phase = 2 * np.pi * frequency * baseline * angle / C_LIGHT
    assert np.isclose(recovered_phase, phase_limit)
    print(f'{name:15s} {angle_mas:14.3g} {delay_ps:11.3g} {ut1_bound:14.3g}')


三类示例的位置误差上限分别约为 34 mas、1.07 mas 和 0.0409 mas；延迟误差上限分别约为 19.8 ps、0.278 ps 和 3.31 ps；最不利投影下的 UT1 误差上限约为 $2.27\,\mathrm{ms}$、$71.4\,\mu\mathrm{s}$ 和 $2.72\,\mu\mathrm{s}$。这些结果只表示由给定基线、频率和 $10^\circ$ 相位门限得到的单项误差预算，不是观测设施的技术规范。

常规成像软件必须正确读取时间标准和 EOP，不能以 UTC 数值直接代替 UT1。精密天体测量还必须证明残余误差在目标源与校准源的差分处理中得到控制。条纹拟合可以吸收部分台站钟差和几何模型残差，但不能修复错误的参考架、未建模的源结构或缺失的绝对位置基准。

![相位参考误差预算](figures/phase_reference_error_budget.png)

**图 3.5.3**：相位参考天体测量的主要误差项。连通阵列和 VLBI 均受热噪声、校准源位置、传播介质和模型误差限制，其中 VLBI 对时钟、地球定向参数和源结构更为敏感。图中数值仅用于表示误差项的相对量级。

### 3.5.5 案例：跨频段相位参考位置偏移的判读

设目标为一颗紧致活动星系核，相位校准源与目标相距约 $1^\circ$。若低频和高频图像的峰值位置不重合，不能立即将其解释为天体结构的真实变化。首先应检查两个频段的相位中心、频率坐标和参考频率是否一致，成像是否采用相同的天球参考系统与投影，校准源位置是否来自同一参考架，以及相位参考循环时间和解算时间间隔是否满足相干性要求。

其次，应区分三类误差来源。几何与时钟模型误差通常表现为随基线、时间或频率变化的规则相位斜率。传播介质误差中，电离层效应具有色散性并在低频更强；中性大气路径近似非色散，但同一路径误差产生的相位随频率增大，因而限制高频相干时间。源结构效应则包括目标源的频率相关核心位移，以及校准源偏离理想点源所产生的结构相位。若未区分这些效应便直接配准跨频段图像，谱指数和喷流结构测量可能混入校准系统误差。

该案例说明，位置天文学中的时间、坐标和参考架约定会直接影响后续校准及科学解释。第 8 章将进一步讨论校准模型，后续 VLBI 与高频观测内容将继续使用本节的延迟和误差预算关系。

### 3.5.6 本节小结

* UTC、TAI、TT 和 UT1 承担不同的计时功能，UT1 与 EOP 共同确定地球自转和姿态。
* ICRS/ICRF 与 ITRS/ITRF 分别表示天球和地球参考系统及其参考架实现，二者不能混称为单一“参考系”。
* ECEF 和 ENU 是坐标表示形式，坐标数值仍必须附带参考架、历元和方向约定。
* 延迟误差通过 $|\Delta\phi|=2\pi\nu|\Delta\tau|$ 转化为相位误差，并通过 $|\Delta\theta|\simeq c|\Delta\tau|/B$ 限制天体测量精度。
* 高频相位参考、VLBI 和精密位置测量必须记录参考架版本、参数历元、EOP 来源、延迟模型版本和相位跟踪中心。

***

下一节：[3.P 综合问题集](3_problem_set.ipynb)
